
# Bioreactor ML Workflow — Import, Clean, Model (5 models) & Visualize

This notebook loads `data/bioreactor_ml_dataset.csv`, performs light data cleaning, trains **five regression models** to predict `Product_Titer_gL`, evaluates them, and generates visualizations.

**Models:** Linear Regression, RidgeCV, Random Forest, Gradient Boosting, SVR (RBF)

> Tip: Run cells **top to bottom**. Outputs (plots and cleaned data) will be saved in an `outputs/` folder.


In [ ]:
# 1) Setup: imports & options
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.inspection import permutation_importance
from pathlib import Path

# Plot style
sns.set(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paths
DATA_PATH = Path('../data/bioreactor_ml_dataset.csv')
OUTPUT_DIR = Path('../models'); OUTPUT_DIR.mkdir(exist_ok=True)
CLEAN_PATH = Path('../data/bioreactor_ml_dataset_clean.csv')

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from pathlib import Path

# Path to your dataset
DATA_PATH = Path("../data/bioreactor_ml_dataset.csv")
assert DATA_PATH.exists(), f"Data file not found: {DATA_PATH.resolve()}"

# Load data
df = pd.read_csv(DATA_PATH)
print("Original shape:", df.shape)
print("Columns found:", df.columns.tolist())

# --- Save a backup copy of the original file ---
backup_path = DATA_PATH.parent / "bioreactor_ml_dataset_Original.csv"
df.to_csv(backup_path, index=False)
print("📂 Original dataset saved as:", backup_path.resolve())

# Clean column names
df.columns = df.columns.str.strip()

# Drop incomplete rows
df = df.dropna()

# Derived features (before vs after changes)
df["Glucose_Consumption_Rate"] = df["Glucose_gL"].diff()
df["DO_Change"] = df["Dissolved_Oxygen_percent"].diff()
df["Specific_Productivity"] = df["Product_Titer_gL"] / df["Cell_Viability_percent"]

# Normalize agitation
scaler = StandardScaler()
df["Agitation_Normalized"] = scaler.fit_transform(df[["Agitation_RPM"]])

# Flags
df["High_Titer_Flag"] = (df["Product_Titer_gL"] > 1).astype(int)
df["Low_Viability_Flag"] = (df["Cell_Viability_percent"] < 98).astype(int)

# --- Override the same file with cleaned dataset ---
df.to_csv(DATA_PATH, index=False)

print("✅ Cleaned dataset saved (overwritten):", DATA_PATH.resolve())
print("New shape:", df.shape)
print(df.head(5))


In [ ]:
# 3) Quick inspection
print('\nDtypes:')
print(df.dtypes)
print('\nMissing values per column:')
print(df.isna().sum())
print('\nDuplicates:', df.duplicated().sum())
df.describe()

In [ ]:
# 4) Data cleaning: drop duplicates; ensure numeric types
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Dropped {before - len(df)} duplicate rows")

for col in df.columns:
    if not np.issubdtype(df[col].dtype, np.number):
        df[col] = pd.to_numeric(df[col], errors='coerce')

print('Missing after coercion:\n', df.isna().sum())

# Save a cleaned copy
df.to_csv(CLEAN_PATH, index=False)
print('Cleaned dataset saved to:', CLEAN_PATH.resolve())

In [ ]:
# 5) Correlation heatmap
plt.figure(figsize=(10,7))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation heatmap (bioreactor variables)')
plt.tight_layout()
heatmap_path = OUTPUT_DIR / 'correlation_heatmap.png'
plt.savefig(heatmap_path)
print('Saved:', heatmap_path.resolve())
plt.show()

In [ ]:
# 6) Target distribution
plt.figure(figsize=(7,4))
sns.histplot(df['Product_Titer_gL'], bins=40, kde=True)
plt.xlabel('Product_Titer_gL')
plt.title('Distribution of Product Titer (g/L)')
plt.tight_layout()
dist_path = OUTPUT_DIR / 'target_distribution.png'
plt.savefig(dist_path)
print('Saved:', dist_path.resolve())
plt.show()

In [ ]:
# 7) Product titer over time
if 'Time_hours' in df.columns:
    plt.figure(figsize=(9,4))
    sns.lineplot(x=df['Time_hours'], y=df['Product_Titer_gL'])
    plt.xlabel('Time (hours)')
    plt.ylabel('Product_Titer_gL')
    plt.title('Product Titer over Time')
    plt.tight_layout()
    ts_path = OUTPUT_DIR / 'titer_over_time.png'
    plt.savefig(ts_path)
    print('Saved:', ts_path.resolve())
    plt.show()
else:
    print('Column Time_hours not found; skipping time series plot.')

In [ ]:
# 8) Train-test split
TARGET = 'Product_Titer_gL'
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
X_train.shape, X_test.shape

In [ ]:
# 9) Define 5 models in pipelines
numeric_imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

models = {
    'LinearRegression': Pipeline([
        ('imputer', numeric_imputer),
        ('scaler', scaler),
        ('model', LinearRegression())
    ]),
    'RidgeCV': Pipeline([
        ('imputer', numeric_imputer),
        ('scaler', scaler),
        ('model', RidgeCV(alphas=np.logspace(-3, 3, 13), cv=5))
    ]),
    'RandomForestRegressor': Pipeline([
        ('imputer', numeric_imputer),
        ('model', RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE))
    ]),
    'GradientBoostingRegressor': Pipeline([
        ('imputer', numeric_imputer),
        ('model', GradientBoostingRegressor(random_state=RANDOM_STATE))
    ]),
    'SVR': Pipeline([
        ('imputer', numeric_imputer),
        ('scaler', scaler),
        ('model', SVR(kernel='rbf', C=10.0, epsilon=0.1))
    ])
}
print(list(models.keys()))

In [ ]:
# 10) Train, cross-validate, and evaluate on test set
from math import sqrt

results = []
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for name, pipe in models.items():
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='r2')
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = sqrt(mean_squared_error(y_test, y_pred))
    results.append({
        'model': name,
        'cv_r2_mean': cv_scores.mean(),
        'cv_r2_std': cv_scores.std(),
        'test_r2': r2,
        'test_mae': mae,
        'test_rmse': rmse
    })

results_df = pd.DataFrame(results).sort_values(by='test_r2', ascending=False)
print(results_df)
results_df

In [ ]:
# 11) Pick best model (highest test R^2) and plot Actual vs Predicted
best_model_name = results_df.iloc[0]['model']
best_pipe = models[best_model_name]

# Ensure best is fitted
best_pipe.fit(X_train, y_train)
y_pred_best = best_pipe.predict(X_test)

plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred_best, alpha=0.6)
lims = [min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())]
plt.plot(lims, lims, 'r--', label='Ideal')
plt.xlabel('Actual Product_Titer_gL')
plt.ylabel('Predicted Product_Titer_gL')
plt.title(f'Actual vs Predicted — {best_model_name}')
plt.legend()
plt.tight_layout()
avp_path = OUTPUT_DIR / 'actual_vs_pred_best.png'
plt.savefig(avp_path)
print('Best model:', best_model_name)
print('Saved:', avp_path.resolve())
plt.show()

In [ ]:

# 12) Feature importance (tree-based) or permutation importance otherwise
fi_path = None
model_step = best_pipe.named_steps.get('model')

try:
    if hasattr(model_step, 'feature_importances_'):
        # Refit to ensure feature_importances_ present
        best_pipe.fit(X_train, y_train)
        importances = model_step.feature_importances_
        fi = pd.Series(importances, index=X.columns).sort_values(ascending=False)
        plt.figure(figsize=(8,5))
        sns.barplot(x=fi.values, y=fi.index)
        plt.xlabel('Feature importance')
        plt.ylabel('Feature')
        plt.title(f'Feature importance — {best_model_name}')
        plt.tight_layout()
        fi_path = OUTPUT_DIR / 'feature_importance.png'
        plt.savefig(fi_path)
        print('Saved:', fi_path.resolve())
        plt.show()
    else:
        r = permutation_importance(best_pipe, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE)
        fi = pd.Series(r.importances_mean, index=X.columns).sort_values(ascending=False)
        plt.figure(figsize=(8,5))
        sns.barplot(x=fi.values, y=fi.index)
        plt.xlabel('Permutation importance (mean)')
        plt.ylabel('Feature')
        plt.title(f'Permutation importance — {best_model_name}')
        plt.tight_layout()
        fi_path = OUTPUT_DIR / 'feature_importance_permutation.png'
        plt.savefig(fi_path)
        print('Saved:', fi_path.resolve())
        plt.show()
except Exception as e:
    print('Importance computation failed:', e)


In [ ]:
# 13) (Optional) Time-based evaluation: train on early times, test on later times
import math
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

def compute_rmse(y_true, y_pred):
    """
    Version-safe RMSE:
      1) Try sklearn >= 1.6: root_mean_squared_error
      2) Try mean_squared_error(..., squared=False)
      3) Fallback: sqrt(mean_squared_error(...))
    """
    try:
        from sklearn.metrics import root_mean_squared_error
        return float(root_mean_squared_error(y_true, y_pred))
    except Exception:
        pass
    try:
        return float(mean_squared_error(y_true, y_pred, squared=False))
    except TypeError:
        return float(math.sqrt(mean_squared_error(y_true, y_pred)))

if 'Time_hours' in df.columns:
    cutoff = int(df['Time_hours'].quantile(0.8))  # 80% time cutoff
    train_mask = df['Time_hours'] <= cutoff
    X_tr, y_tr = df.loc[train_mask].drop(columns=['Product_Titer_gL']), df.loc[train_mask, 'Product_Titer_gL']
    X_te, y_te = df.loc[~train_mask].drop(columns=['Product_Titer_gL']), df.loc[~train_mask, 'Product_Titer_gL']

    best_pipe.fit(X_tr, y_tr)
    pred_te = best_pipe.predict(X_te)

    r2_t  = r2_score(y_te, pred_te)
    mae_t = mean_absolute_error(y_te, pred_te)
    rmse_t = compute_rmse(y_te, pred_te)

    print(f'Time-based split (<= {cutoff} hrs train, > {cutoff} hrs test):')
    print({'r2': r2_t, 'mae': mae_t, 'rmse': rmse_t})
else:
    print('Time_hours not found; skipping time-based evaluation.')

In [ ]:

# 14) (Optional) Refit excluding Time_hours to emphasize process knobs
if 'Time_hours' in X.columns:
    X_nt = df.drop(columns=['Product_Titer_gL', 'Time_hours'])
    y_nt = df['Product_Titer_gL']
    X_tr_nt, X_te_nt, y_tr_nt, y_te_nt = train_test_split(X_nt, y_nt, test_size=0.2, random_state=RANDOM_STATE)

    pipe_nt = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', RidgeCV(alphas=np.logspace(-3,3,13), cv=5))
    ])

    pipe_nt.fit(X_tr_nt, y_tr_nt)
    pred_nt = pipe_nt.predict(X_te_nt)
    print('R^2 (no Time_hours):', r2_score(y_te_nt, pred_nt))
    r = permutation_importance(pipe_nt, X_te_nt, y_te_nt, n_repeats=10, random_state=RANDOM_STATE)
    fi_nt = pd.Series(r.importances_mean, index=X_nt.columns).sort_values(ascending=False)

    plt.figure(figsize=(8,5))
    sns.barplot(x=fi_nt.values, y=fi_nt.index)
    plt.xlabel('Permutation importance (mean)')
    plt.ylabel('Feature')
    plt.title('Permutation importance — Ridge (no Time_hours)')
    plt.tight_layout()
    fi_nt_path = OUTPUT_DIR / 'feature_importance_no_time.png'
    plt.savefig(fi_nt_path)
    print('Saved:', fi_nt_path.resolve())
    plt.show()
else:
    print('Time_hours not present; skipping this section.')


In [ ]:
pip install streamlit numpy pandas scikit-learn joblib matplotlib seaborn